## 1. Setup

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet50, vit_b_16
from torchvision.transforms import InterpolationMode

torch.set_num_threads(os.cpu_count() or 8)

ROOT = Path.cwd()
if not (ROOT / "runs").exists():
    fallback = Path(r"C:\Users\dylan\OneDrive - Swinburne Sarawak\Desktop\evaluation")
    if (fallback / "runs").exists():
        ROOT = fallback

DATA_ROOT = Path(r"C:\Users\dylan\Downloads\DPD_Small")
OUTPUT_DIR = ROOT / "outputs"
REUSE_LOGITS = True  # set False to run every network again

NUM_CROPS = 55
NUM_DISEASES = 175
NUM_COMPOSITIONS = 333
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def _spec(name, label, backbone, setup, family, kind, checkpoint, metrics, batch_size):
    return {
        "name": name,
        "label": label,
        "backbone": backbone,
        "setup": setup,
        "family": family,
        "kind": kind,
        "checkpoint": ROOT / checkpoint,
        "metrics": None if metrics is None else ROOT / metrics,
        "batch_size": batch_size,
    }


MODELS = [
    _spec(
        "independent_resnet50",
        "Baseline 1: independent ResNet-50",
        "ResNet-50",
        "independent",
        "resnet",
        "independent_resnet",
        Path("final-20261111T151540Z-1-001") / "final" / "best.pt",
        Path("final-20261111T151540Z-1-001") / "final" / "metrics.json",
        16,
    ),
    _spec(
        "multitask_resnet50",
        "Baseline 2: multi-task ResNet-50",
        "ResNet-50",
        "multi-task",
        "resnet",
        "multitask_resnet",
        Path("final-20261111T151317Z-1-001") / "final" / "best.pt",
        Path("final-20261111T151317Z-1-001") / "final" / "metrics.json",
        32,
    ),
    _spec(
        "independent_vit_b16",
        "Baseline 1: independent ViT-B/16",
        "ViT-B/16",
        "independent",
        "vit",
        "independent_vit",
        Path("runs") / "runs" / "final_best.pt",
        Path("runs") / "runs" / "metrics.json",
        8,
    ),
    _spec(
        "multitask_vit_b16",
        "Baseline 2: multi-task ViT-B/16",
        "ViT-B/16",
        "multi-task",
        "vit",
        "multitask_vit",
        Path("baseline2_multitask_vit-20261113T094621Z-1-001") / "baseline2_multitask_vit" / "dev" / "best.pt",
        None,
        8,
    ),
]
MODEL_BY_NAME = {spec["name"]: spec for spec in MODELS}

# Leave this as all four keys, or pass a subset such as ["independent_resnet50"].
MODELS_TO_RUN = [spec["name"] for spec in MODELS]
print("project:", ROOT)
print("data:", DATA_ROOT)
print("models:", ", ".join(MODELS_TO_RUN))


project: c:\Users\dylan\OneDrive - Swinburne Sarawak\Desktop\evaluation
data: C:\Users\dylan\Downloads\DPD_Small
models: independent_resnet50, multitask_resnet50, independent_vit_b16, multitask_vit_b16


## 2. Architectures and weight loading

Each class matches the keys in its checkpoint. `load_baseline` builds that class, loads the weights, and refuses to continue if any key is missing or unexpected.


In [2]:
def eval_transform(family: str) -> transforms.Compose:
    interpolation = InterpolationMode.BILINEAR if family == "resnet" else InterpolationMode.BICUBIC
    return transforms.Compose([
        transforms.Resize((224, 224), interpolation=interpolation),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def _resnet_backbone() -> nn.Module:
    backbone = resnet50(weights=None)
    backbone.fc = nn.Identity()
    return backbone


def _vit_backbone() -> nn.Module:
    backbone = vit_b_16(weights=None)
    backbone.heads = nn.Identity()
    return backbone


class IndependentResNet(nn.Module):
    """Two ResNet-50 networks, one crop head and one disease head."""

    def __init__(self) -> None:
        super().__init__()
        self.crop = nn.Module()
        self.crop.backbone = _resnet_backbone()
        self.crop.head = nn.Linear(2048, NUM_CROPS)
        self.disease = nn.Module()
        self.disease.backbone = _resnet_backbone()
        self.disease.head = nn.Linear(2048, NUM_DISEASES)

    def forward(self, images: torch.Tensor):
        crop = self.crop.head(self.crop.backbone(images))
        disease = self.disease.head(self.disease.backbone(images))
        return crop, disease, None


class MultiTaskResNet(nn.Module):
    """One shared ResNet-50 with a crop head and a disease head."""

    def __init__(self) -> None:
        super().__init__()
        self.backbone = _resnet_backbone()
        self.crop_head = nn.Linear(2048, NUM_CROPS)
        self.disease_head = nn.Linear(2048, NUM_DISEASES)

    def forward(self, images: torch.Tensor):
        features = self.backbone(images)
        return self.crop_head(features), self.disease_head(features), None


class IndependentViT(nn.Module):
    """Two ViT-B/16 networks, one crop head and one disease head."""

    def __init__(self) -> None:
        super().__init__()
        self.crop = nn.Module()
        self.crop.backbone = _vit_backbone()
        self.crop.head = nn.Linear(768, NUM_CROPS)
        self.disease = nn.Module()
        self.disease.backbone = _vit_backbone()
        self.disease.head = nn.Linear(768, NUM_DISEASES)

    def forward(self, images: torch.Tensor):
        crop = self.crop.head(self.crop.backbone(images))
        disease = self.disease.head(self.disease.backbone(images))
        return crop, disease, None


class _TimmAttention(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.num_heads = 12
        self.head_dim = 64
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(768, 2304)
        self.proj = nn.Linear(768, 768)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, tokens, channels = x.shape
        qkv = self.qkv(x).reshape(batch, tokens, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        query, key, value = qkv.unbind(0)
        attention = (query @ key.transpose(-2, -1)) * self.scale
        attention = attention.softmax(dim=-1)
        mixed = (attention @ value).transpose(1, 2).reshape(batch, tokens, channels)
        return self.proj(mixed)


class _TimmBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(768, eps=1e-6)
        self.attn = _TimmAttention()
        self.norm2 = nn.LayerNorm(768, eps=1e-6)
        self.mlp = nn.Module()
        self.mlp.fc1 = nn.Linear(768, 3072)
        self.mlp.fc2 = nn.Linear(3072, 768)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        hidden = nn.functional.gelu(self.mlp.fc1(self.norm2(x)))
        return x + self.mlp.fc2(hidden)


class _TimmViT(nn.Module):
    """timm vit_base_patch16_224, class token only, matching dev/best.pt."""

    def __init__(self) -> None:
        super().__init__()
        self.cls_token = nn.Parameter(torch.zeros(1, 1, 768))
        self.pos_embed = nn.Parameter(torch.zeros(1, 197, 768))
        self.patch_embed = nn.Module()
        self.patch_embed.proj = nn.Conv2d(3, 768, kernel_size=16, stride=16)
        self.blocks = nn.Sequential(*[_TimmBlock() for _ in range(12)])
        self.norm = nn.LayerNorm(768, eps=1e-6)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed.proj(x).flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls, x), dim=1) + self.pos_embed
        return self.norm(self.blocks(x))[:, 0]


class MultiTaskViT(nn.Module):
    """One shared timm ViT-B/16 with a crop head and a disease head."""

    def __init__(self) -> None:
        super().__init__()
        self.backbone = _TimmViT()
        self.crop_head = nn.Linear(768, NUM_CROPS)
        self.disease_head = nn.Linear(768, NUM_DISEASES)

    def forward(self, images: torch.Tensor):
        features = self.backbone(images)
        return self.crop_head(features), self.disease_head(features), None


BUILDERS = {
    "independent_resnet": IndependentResNet,
    "multitask_resnet": MultiTaskResNet,
    "independent_vit": IndependentViT,
    "multitask_vit": MultiTaskViT,
}


def unpack_checkpoint(blob):
    """Return weights, epoch, stored bias, and metrics saved beside the weights."""
    epoch = None
    stored_bias = None
    stored_metrics = None
    if isinstance(blob, dict) and "model_state_dict" in blob:
        state = blob["model_state_dict"]
        epoch = blob.get("epoch")
        stored_bias = blob.get("bias")
        stored_metrics = blob.get("metrics")
    elif isinstance(blob, dict) and "models" in blob:
        state = blob["models"]
        epoch = blob.get("epoch")
        stored_bias = blob.get("bias")
    elif isinstance(blob, dict) and "model" in blob and not torch.is_tensor(blob["model"]):
        state = blob["model"]
        epoch = blob.get("epoch")
        stored_bias = blob.get("bias")
    else:
        state = blob
    return state, epoch, stored_bias, stored_metrics


def fraction_metrics(metrics) -> dict:
    """Return metric values as fractions. Numbers above 1.5 are percentages."""
    if not isinstance(metrics, dict):
        return {}
    values = {
        str(key): float(value)
        for key, value in metrics.items()
        if isinstance(value, (int, float))
    }
    if not values:
        return {}
    scale = 100.0 if max(values.values()) > 1.5 else 1.0
    return {key: value / scale for key, value in values.items()}


def load_baseline(name: str) -> tuple[nn.Module, dict]:
    """Build one baseline and load its checkpoint. The module is left in eval mode."""
    spec = MODEL_BY_NAME[name]
    checkpoint = spec["checkpoint"]
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)

    blob = torch.load(checkpoint, map_location="cpu", weights_only=False)
    state, epoch, stored_bias, stored_metrics = unpack_checkpoint(blob)

    model = BUILDERS[spec["kind"]]()
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        raise RuntimeError(
            f"{name} did not match the checkpoint "
            f"(missing {len(missing)}, unexpected {len(unexpected)})"
        )
    model.eval()
    meta = {
        "epoch": None if epoch is None else int(epoch),
        "stored_bias": None if stored_bias is None else float(stored_bias),
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "checkpoint": str(checkpoint),
        "stored_metrics": stored_metrics,
    }
    return model, meta


## 3. Data and forward pass

The CSVs have no header. Columns are filename, crop id, disease id, composition id, then the three names. Images are resized to 224 with bilinear interpolation for ResNet and bicubic for ViT, then ImageNet-normalised.


In [3]:
class SplitDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, image_dir: Path, transform) -> None:
        self.frame = frame.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        path = self.image_dir / str(row.iloc[0])
        with Image.open(path) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, int(row.iloc[1]), int(row.iloc[2]), int(row.iloc[3])


def load_split_frames(data_root: Path):
    image_dir = data_root / "DPD_Small_dataset"
    paths = {
        "train": data_root / "DPD_Small_train.csv",
        "seen": data_root / "DPD_Small_test_seen.csv",
        "unseen": data_root / "DPD_Small_test_unseen.csv",
    }
    missing = [str(path) for path in (*paths.values(), image_dir) if not path.exists()]
    if missing:
        raise FileNotFoundError("DPD-Small files not found:\n" + "\n".join(missing))
    frames = {key: pd.read_csv(path, header=None) for key, path in paths.items()}
    return frames["train"], frames["seen"], frames["unseen"]


def composition_index(train: pd.DataFrame, seen: pd.DataFrame, unseen: pd.DataFrame):
    """Map each of the 333 composition ids to its crop, disease, and seen flag."""
    crop_of = np.full(NUM_COMPOSITIONS, -1, dtype=np.int64)
    disease_of = np.full(NUM_COMPOSITIONS, -1, dtype=np.int64)
    for frame in (train, seen, unseen):
        for comp, crop, disease in zip(frame[3].to_numpy(), frame[1].to_numpy(), frame[2].to_numpy()):
            comp = int(comp)
            if crop_of[comp] == -1:
                crop_of[comp] = int(crop)
                disease_of[comp] = int(disease)
            elif crop_of[comp] != int(crop) or disease_of[comp] != int(disease):
                raise ValueError(f"composition {comp} maps to two different crop/disease pairs")
    if (crop_of < 0).any():
        missing_ids = np.flatnonzero(crop_of < 0).tolist()
        raise ValueError(f"missing composition ids: {missing_ids[:10]}")
    seen_mask = np.zeros(NUM_COMPOSITIONS, dtype=bool)
    seen_mask[train[3].to_numpy().astype(int)] = True
    return crop_of, disease_of, seen_mask


@torch.inference_mode()
def collect_logits(model, frame, image_dir, transform, batch_size, split_name):
    loader = DataLoader(
        SplitDataset(frame, image_dir, transform),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    crop_parts, disease_parts, comp_parts = [], [], []
    crop_y, disease_y, comp_y = [], [], []
    seen_count = 0
    total = len(frame)
    for images, crop, disease, comp in loader:
        crop_logits, disease_logits, comp_logits = model(images)
        crop_parts.append(crop_logits.float().cpu())
        disease_parts.append(disease_logits.float().cpu())
        if comp_logits is not None:
            comp_parts.append(comp_logits.float().cpu())
        crop_y.append(crop)
        disease_y.append(disease)
        comp_y.append(comp)
        seen_count += len(crop)
        print(f"  {split_name} {seen_count}/{total}", flush=True)
    return {
        "crop_logits": torch.cat(crop_parts).numpy(),
        "disease_logits": torch.cat(disease_parts).numpy(),
        "comp_logits": torch.cat(comp_parts).numpy() if comp_parts else np.empty((0, 0)),
        "crop": torch.cat(crop_y).numpy(),
        "disease": torch.cat(disease_y).numpy(),
        "comp": torch.cat(comp_y).numpy(),
    }


def logits_path(name: str) -> Path:
    return OUTPUT_DIR / "logits" / f"{name}.npz"


def save_logits(name: str, seen: dict, unseen: dict) -> None:
    path = logits_path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {}
    for split, blob in (("seen", seen), ("unseen", unseen)):
        for key, value in blob.items():
            payload[f"{split}_{key}"] = value
    np.savez_compressed(path, **payload)


def load_logits(name: str):
    blob = np.load(logits_path(name))
    keys = ("crop_logits", "disease_logits", "comp_logits", "crop", "disease", "comp")

    def take(split: str) -> dict:
        return {key: blob[f"{split}_{key}"] for key in keys}

    return take("seen"), take("unseen")


## 4. Scores

Pair top-1 and top-5 are computed inside the closed set of 333 compositions. The reported bias, when the checkpoint has one, is applied as `score -= bias` on seen compositions. The oracle bias is the value on this test sweep with the best harmonic mean. It is an upper bound, not the number to quote. AUC is the area under unseen top-1 plotted against seen top-1.


In [4]:
def pair_scores(crop_logits, disease_logits, crop_of, disease_of, seen_mask, bias: float):
    scores = crop_logits[:, crop_of] + disease_logits[:, disease_of]
    if bias != 0.0:
        scores = scores.copy()
        scores[:, seen_mask] -= bias
    return scores


def topk_accuracy(scores: np.ndarray, targets: np.ndarray, k: int = 5):
    if len(targets) == 0:
        return 0.0, 0.0, np.empty(0, dtype=np.int64)
    predicted = scores.argmax(axis=1)
    top1 = float((predicted == targets).mean())
    chosen = np.argpartition(scores, -k, axis=1)[:, -k:]
    top5 = float((chosen == targets[:, None]).any(axis=1).mean())
    return top1, top5, predicted


def harmonic_mean(seen: float, unseen: float) -> float:
    total = seen + unseen
    if total == 0.0:
        return 0.0
    return float(2.0 * seen * unseen / total)


def operating_point(seen_scores, seen_targets, unseen_scores, unseen_targets) -> dict:
    seen_top1, seen_top5, _ = topk_accuracy(seen_scores, seen_targets)
    unseen_top1, unseen_top5, _ = topk_accuracy(unseen_scores, unseen_targets)
    return {
        "seen_top1": seen_top1,
        "seen_top5": seen_top5,
        "unseen_top1": unseen_top1,
        "unseen_top5": unseen_top5,
        "harmonic_mean": harmonic_mean(seen_top1, unseen_top1),
    }


def bias_curve(
    seen_crop, seen_disease, seen_targets,
    unseen_crop, unseen_disease, unseen_targets,
    crop_of, disease_of, seen_mask, extra_biases,
):
    grid = np.linspace(-12.0, 24.0, 361)
    biases = np.unique(np.concatenate([grid, np.asarray(extra_biases, dtype=np.float64)]))
    rows = []
    for bias in biases:
        point = operating_point(
            pair_scores(seen_crop, seen_disease, crop_of, disease_of, seen_mask, float(bias)),
            seen_targets,
            pair_scores(unseen_crop, unseen_disease, crop_of, disease_of, seen_mask, float(bias)),
            unseen_targets,
        )
        rows.append({"bias": float(bias), **point})
    curve = pd.DataFrame(rows)
    order = np.argsort(curve["seen_top1"].to_numpy(), kind="mergesort")
    area = float(np.trapezoid(curve["unseen_top1"].to_numpy()[order], curve["seen_top1"].to_numpy()[order]))
    best = curve.loc[curve["harmonic_mean"].idxmax()]
    summary = {
        "auc": area,
        "oracle_bias": float(best["bias"]),
        "oracle_harmonic_mean": float(best["harmonic_mean"]),
        "oracle_seen_top1": float(best["seen_top1"]),
        "oracle_unseen_top1": float(best["unseen_top1"]),
    }
    return curve, summary


def head_accuracy(crop_logits, disease_logits, crop_y, disease_y) -> dict:
    crop_hit = crop_logits.argmax(axis=1) == crop_y
    disease_hit = disease_logits.argmax(axis=1) == disease_y
    return {
        "crop": float(crop_hit.mean()),
        "disease": float(disease_hit.mean()),
        "both": float((crop_hit & disease_hit).mean()),
    }


def shortcut_rate(predicted_pairs: np.ndarray, seen_mask: np.ndarray) -> float:
    """Share of unseen images whose top pair, at bias 0, is a training composition."""
    if len(predicted_pairs) == 0:
        return 0.0
    return float(seen_mask[predicted_pairs].mean())


def score_logits(seen, unseen, crop_of, disease_of, seen_mask, dev_bias):
    extras = [0.0] if dev_bias is None else [0.0, float(dev_bias)]
    curve, summary = bias_curve(
        seen["crop_logits"], seen["disease_logits"], seen["comp"],
        unseen["crop_logits"], unseen["disease_logits"], unseen["comp"],
        crop_of, disease_of, seen_mask, extras,
    )
    zero = operating_point(
        pair_scores(seen["crop_logits"], seen["disease_logits"], crop_of, disease_of, seen_mask, 0.0),
        seen["comp"],
        pair_scores(unseen["crop_logits"], unseen["disease_logits"], crop_of, disease_of, seen_mask, 0.0),
        unseen["comp"],
    )
    _, _, unseen_pred = topk_accuracy(
        pair_scores(unseen["crop_logits"], unseen["disease_logits"], crop_of, disease_of, seen_mask, 0.0),
        unseen["comp"],
    )
    seen_heads = head_accuracy(seen["crop_logits"], seen["disease_logits"], seen["crop"], seen["disease"])
    unseen_heads = head_accuracy(unseen["crop_logits"], unseen["disease_logits"], unseen["crop"], unseen["disease"])
    result = {
        "b0": zero,
        "dev_bias": dev_bias,
        "calibrated": None,
        "oracle": summary,
        "auc": summary["auc"],
        "heads": {"seen": seen_heads, "unseen": unseen_heads},
        "paper_harmonic_mean": harmonic_mean(seen_heads["both"], unseen_heads["both"]),
        "shortcut_rate": shortcut_rate(unseen_pred, seen_mask),
        "comp_head": None,
    }
    if dev_bias is not None:
        result["calibrated"] = operating_point(
            pair_scores(seen["crop_logits"], seen["disease_logits"], crop_of, disease_of, seen_mask, dev_bias),
            seen["comp"],
            pair_scores(unseen["crop_logits"], unseen["disease_logits"], crop_of, disease_of, seen_mask, dev_bias),
            unseen["comp"],
        )
        result["calibrated"]["bias"] = dev_bias
    if seen["comp_logits"].size and unseen["comp_logits"].size:
        result["comp_head"] = operating_point(
            seen["comp_logits"], seen["comp"], unseen["comp_logits"], unseen["comp"]
        )
    return result, curve


def dev_bias_for(spec: dict, meta: dict):
    if spec["metrics"] is not None and spec["metrics"].is_file():
        saved = json.loads(spec["metrics"].read_text(encoding="utf-8"))
        if "dev_bias" in saved:
            return float(saved["dev_bias"])
    stored = meta.get("stored_bias")
    return None if stored is None else float(stored)


## 5. Run all four models and build the comparison table

In [5]:
def result_row(spec: dict, meta: dict, scored: dict) -> dict:
    calibrated = scored["calibrated"] or {}
    comp_head = scored["comp_head"] or {}
    stored = fraction_metrics(meta.get("stored_metrics"))
    return {
        "model": spec["label"],
        "key": spec["name"],
        "backbone": spec["backbone"],
        "setup": spec["setup"],
        "checkpoint": meta.get("checkpoint") or str(spec["checkpoint"]),
        "epoch": meta.get("epoch"),
        "parameters": meta.get("parameters"),
        "checkpoint_seen_top1": stored.get("seen_top1_acc"),
        "checkpoint_seen_top5": stored.get("seen_top5_acc"),
        "checkpoint_unseen_top1": stored.get("unseen_top1_acc"),
        "checkpoint_unseen_top5": stored.get("unseen_top5_acc"),
        "checkpoint_harmonic_mean": stored.get("harmonic_mean"),
        "seen_top1_b0": scored["b0"]["seen_top1"],
        "seen_top5_b0": scored["b0"]["seen_top5"],
        "unseen_top1_b0": scored["b0"]["unseen_top1"],
        "unseen_top5_b0": scored["b0"]["unseen_top5"],
        "harmonic_mean_b0": scored["b0"]["harmonic_mean"],
        "dev_bias": scored["dev_bias"],
        "seen_top1_calibrated": calibrated.get("seen_top1"),
        "seen_top5_calibrated": calibrated.get("seen_top5"),
        "unseen_top1_calibrated": calibrated.get("unseen_top1"),
        "unseen_top5_calibrated": calibrated.get("unseen_top5"),
        "harmonic_mean_calibrated": calibrated.get("harmonic_mean"),
        "oracle_bias": scored["oracle"]["oracle_bias"],
        "harmonic_mean_oracle": scored["oracle"]["oracle_harmonic_mean"],
        "auc": scored["auc"],
        "seen_crop_acc": scored["heads"]["seen"]["crop"],
        "seen_disease_acc": scored["heads"]["seen"]["disease"],
        "seen_both_acc": scored["heads"]["seen"]["both"],
        "unseen_crop_acc": scored["heads"]["unseen"]["crop"],
        "unseen_disease_acc": scored["heads"]["unseen"]["disease"],
        "unseen_both_acc": scored["heads"]["unseen"]["both"],
        "paper_harmonic_mean": scored["paper_harmonic_mean"],
        "shortcut_rate": scored["shortcut_rate"],
        "comp_head_seen_top1": comp_head.get("seen_top1"),
        "comp_head_unseen_top1": comp_head.get("unseen_top1"),
        "comp_head_harmonic_mean": comp_head.get("harmonic_mean"),
    }


def reference_error(spec: dict, scored: dict):
    """Largest gap versus a saved metrics.json, when that file exists."""
    if spec["metrics"] is None or not spec["metrics"].is_file():
        return None
    saved = json.loads(spec["metrics"].read_text(encoding="utf-8"))
    zero = saved["results"]["b = 0 (no calibration)"]
    calibrated = saved["results"]["b from dev validation (REPORTED)"]
    pairs = [
        (scored["b0"]["seen_top1"], zero["seen top-1"]),
        (scored["b0"]["unseen_top1"], zero["unseen top-1"]),
        (scored["b0"]["seen_top5"], zero["seen top-5"]),
        (scored["b0"]["unseen_top5"], zero["unseen top-5"]),
        (scored["b0"]["harmonic_mean"], zero["harmonic mean"]),
        (scored["heads"]["seen"]["crop"], saved["heads"]["test-seen"]["crop acc"]),
        (scored["heads"]["seen"]["disease"], saved["heads"]["test-seen"]["disease acc"]),
        (scored["heads"]["unseen"]["crop"], saved["heads"]["test-unseen"]["crop acc"]),
        (scored["heads"]["unseen"]["disease"], saved["heads"]["test-unseen"]["disease acc"]),
        (scored["paper_harmonic_mean"], saved["paper_protocol_hm"]),
        (scored["shortcut_rate"], saved["shortcut_rate"]),
        (scored["auc"], saved["auc"]),
    ]
    if scored["calibrated"] is not None:
        pairs.extend([
            (scored["calibrated"]["seen_top1"], calibrated["seen top-1"]),
            (scored["calibrated"]["unseen_top1"], calibrated["unseen top-1"]),
            (scored["calibrated"]["harmonic_mean"], calibrated["harmonic mean"]),
        ])
    return float(max(abs(got - expected) for got, expected in pairs))


def evaluate_baseline(name: str, data_root: Path = DATA_ROOT, reuse_logits: bool = REUSE_LOGITS) -> dict:
    """Load one checkpoint, score both test splits, and return one comparison row."""
    spec = MODEL_BY_NAME[name]
    print(f"\n=== {spec['label']} ===", flush=True)
    train, seen_frame, unseen_frame = load_split_frames(data_root)
    crop_of, disease_of, seen_mask = composition_index(train, seen_frame, unseen_frame)
    cached = logits_path(name)
    meta = {
        "epoch": None,
        "stored_bias": None,
        "parameters": None,
        "checkpoint": str(spec["checkpoint"]),
    }
    if reuse_logits and cached.is_file():
        print(f"  reusing logits {cached}", flush=True)
        seen, unseen = load_logits(name)
        if spec["metrics"] is not None and spec["metrics"].is_file():
            saved = json.loads(spec["metrics"].read_text(encoding="utf-8"))
            meta["epoch"] = saved.get("epochs")
        elif spec["checkpoint"].is_file():
            blob = torch.load(spec["checkpoint"], map_location="cpu", weights_only=False)
            _, epoch, stored_bias, stored_metrics = unpack_checkpoint(blob)
            meta["epoch"] = epoch
            meta["stored_bias"] = stored_bias
            meta["stored_metrics"] = stored_metrics
            del blob
    else:
        model, meta = load_baseline(name)
        transform = eval_transform(spec["family"])
        image_dir = data_root / "DPD_Small_dataset"
        print(f"  loaded epoch={meta['epoch']} params={meta['parameters']}", flush=True)
        seen = collect_logits(model, seen_frame, image_dir, transform, spec["batch_size"], "test-seen")
        unseen = collect_logits(model, unseen_frame, image_dir, transform, spec["batch_size"], "test-unseen")
        save_logits(name, seen, unseen)
        del model

    bias = dev_bias_for(spec, meta)
    scored, curve = score_logits(seen, unseen, crop_of, disease_of, seen_mask, bias)
    curve_path = OUTPUT_DIR / "curves" / f"{name}.csv"
    curve_path.parent.mkdir(parents=True, exist_ok=True)
    curve.to_csv(curve_path, index=False)

    row = result_row(spec, meta, scored)
    row["reference_max_abs_error"] = reference_error(spec, scored)
    zero = scored["b0"]
    print(
        f"  b = 0: seen {zero['seen_top1']:.4f}  unseen {zero['unseen_top1']:.4f}  "
        f"HM {zero['harmonic_mean']:.4f}",
        flush=True,
    )
    if scored["calibrated"] is not None:
        point = scored["calibrated"]
        print(
            f"  dev bias {bias:.3f}: seen {point['seen_top1']:.4f}  "
            f"unseen {point['unseen_top1']:.4f}  HM {point['harmonic_mean']:.4f}",
            flush=True,
        )
    print(
        f"  oracle bias {scored['oracle']['oracle_bias']:.3f}  "
        f"HM {scored['oracle']['oracle_harmonic_mean']:.4f}  AUC {scored['auc']:.4f}",
        flush=True,
    )
    if row["reference_max_abs_error"] is not None:
        print(f"  max abs error vs saved metrics: {row['reference_max_abs_error']:.4f}", flush=True)
    return row


def compare_baselines(names=None, data_root: Path = DATA_ROOT, reuse_logits: bool = REUSE_LOGITS) -> pd.DataFrame:
    """Evaluate the requested baselines and write outputs/baseline_comparison.csv."""
    chosen = list(names) if names is not None else [spec["name"] for spec in MODELS]
    unknown = [name for name in chosen if name not in MODEL_BY_NAME]
    if unknown:
        raise KeyError(f"unknown model names: {unknown}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    frame = pd.DataFrame([
        evaluate_baseline(name, data_root=data_root, reuse_logits=reuse_logits)
        for name in chosen
    ])
    frame.to_csv(OUTPUT_DIR / "baseline_comparison.csv", index=False)
    (OUTPUT_DIR / "baseline_comparison.json").write_text(
        frame.to_json(orient="records", indent=2),
        encoding="utf-8",
    )
    print(f"\nwrote {OUTPUT_DIR / 'baseline_comparison.csv'}", flush=True)
    return frame


Run the next cell to load every checkpoint and fill the table.

`harmonic_mean_calibrated` uses the development bias saved with each run. The multi-task ViT checkpoint stores that bias (about 4.99) and no separate score block, so its calibrated columns come from the same pair-scoring rule as the other three. `auc` summarises the whole seen/unseen tradeoff and does not depend on picking one bias. The better backbone is the one with the higher mean AUC across its two setups.


In [6]:
comparison = compare_baselines(MODELS_TO_RUN, data_root=DATA_ROOT, reuse_logits=REUSE_LOGITS)

percent_columns = [
    "seen_top1_b0",
    "unseen_top1_b0",
    "harmonic_mean_b0",
    "seen_top1_calibrated",
    "unseen_top1_calibrated",
    "harmonic_mean_calibrated",
    "harmonic_mean_oracle",
    "seen_crop_acc",
    "seen_disease_acc",
    "unseen_crop_acc",
    "unseen_disease_acc",
    "paper_harmonic_mean",
    "checkpoint_seen_top1",
    "checkpoint_unseen_top1",
    "checkpoint_harmonic_mean",
]
table = comparison[
    ["model", "backbone", "setup", "epoch", "parameters", "auc", *percent_columns]
].copy()
for column in percent_columns:
    table[column] = table[column].map(lambda value: "" if pd.isna(value) else f"{value:.1%}")
table["auc"] = comparison["auc"].map(lambda value: f"{value:.3f}")
table["parameters"] = comparison["parameters"].map(
    lambda value: "" if pd.isna(value) else f"{int(value) / 1e6:.1f}M"
)
table



=== Baseline 1: independent ResNet-50 ===
  loaded epoch=16 params=47487334
  test-seen 16/2000
  test-seen 32/2000
  test-seen 48/2000
  test-seen 64/2000
  test-seen 80/2000
  test-seen 96/2000
  test-seen 112/2000
  test-seen 128/2000
  test-seen 144/2000
  test-seen 160/2000
  test-seen 176/2000
  test-seen 192/2000
  test-seen 208/2000
  test-seen 224/2000
  test-seen 240/2000
  test-seen 256/2000
  test-seen 272/2000
  test-seen 288/2000
  test-seen 304/2000
  test-seen 320/2000
  test-seen 336/2000
  test-seen 352/2000
  test-seen 368/2000
  test-seen 384/2000
  test-seen 400/2000
  test-seen 416/2000
  test-seen 432/2000
  test-seen 448/2000
  test-seen 464/2000
  test-seen 480/2000
  test-seen 496/2000
  test-seen 512/2000
  test-seen 528/2000
  test-seen 544/2000
  test-seen 560/2000
  test-seen 576/2000
  test-seen 592/2000
  test-seen 608/2000
  test-seen 624/2000
  test-seen 640/2000
  test-seen 656/2000
  test-seen 672/2000
  test-seen 688/2000
  test-seen 704/2000
  tes

,model,backbone,setup,epoch,parameters,auc,seen_top1_b0,unseen_top1_b0,harmonic_mean_b0,seen_top1_calibrated,...,harmonic_mean_calibrated,harmonic_mean_oracle,seen_crop_acc,seen_disease_acc,unseen_crop_acc,unseen_disease_acc,paper_harmonic_mean,checkpoint_seen_top1,checkpoint_unseen_top1,checkpoint_harmonic_mean
0,Baseline 1: independent ResNet-50,ResNet-50,independent,16,47.5M,0.387,61.1%,6.3%,11.4%,57.0%,...,46.2%,52.9%,85.5%,63.0%,65.4%,6.7%,6.0%,,,
1,Baseline 2: multi-task ResNet-50,ResNet-50,multi-task,17,24.0M,0.364,60.1%,2.6%,5.0%,54.2%,...,47.3%,52.1%,85.5%,62.7%,62.5%,6.2%,3.2%,,,
2,Baseline 1: independent ViT-B/16,ViT-B/16,independent,10,171.8M,0.504,66.9%,10.6%,18.3%,58.8%,...,62.4%,63.4%,89.7%,68.2%,71.1%,11.8%,13.6%,,,
3,Baseline 2: multi-task ViT-B/16,ViT-B/16,multi-task,17,86.0M,0.497,66.6%,4.4%,8.3%,62.1%,...,55.4%,61.3%,92.8%,68.2%,77.3%,7.7%,4.0%,,,


In [7]:
backbone_summary = (
    comparison.groupby("backbone", as_index=False)[["auc", "harmonic_mean_b0", "paper_harmonic_mean"]]
    .mean()
    .sort_values("auc", ascending=False)
)
best_backbone = backbone_summary.iloc[0]["backbone"]
print(f"Better backbone for the deep learning model: {best_backbone}")
print(
    "Chosen by mean AUC of unseen top-1 against seen top-1, "
    "averaged over the independent and multi-task setups."
)
backbone_summary


Better backbone for the deep learning model: ViT-B/16
Chosen by mean AUC of unseen top-1 against seen top-1, averaged over the independent and multi-task setups.


,backbone,auc,harmonic_mean_b0,paper_harmonic_mean
1,ViT-B/16,0.500595,0.132701,0.088116
0,ResNet-50,0.375466,0.082129,0.046260
